# Week 45, Day 4: Priors, urgency and collapsing bounds

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w045/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Where the bias lives

Complete ddm(): each 2 ms step adds drift plus noise to undecided trials and stops those that cross plus or minus bound(t). Then compute the proportion of 'right' (+1) choices in each response time tercile for a starting-point bias (z = 0.3) and a drift bias (+0.4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ddm(v, bound, z=0.0, n=6000, dt=0.002, T=4.0, seed=0):
    g = np.random.default_rng(seed)
    x = np.full(n, z, float); done = np.zeros(n, bool); ch = np.zeros(n, int); rt = np.full(n, T)
    for k in range(int(T / dt)):
        t = (k + 1) * dt; idx = ~done
        ...
        B = bound(t); up = idx & (x >= B); lo = idx & (x <= -B)
        ch[up] = 1; ch[lo] = -1; rt[up | lo] = t; done |= up | lo
        if done.all():
            break
    ch[~done] = np.where(x[~done] >= 0, 1, -1)
    return ch, rt

def by_tercile(values, rt):
    q = np.quantile(rt, [1 / 3, 2 / 3])
    return np.array([values[m].mean() for m in [rt <= q[0], (rt > q[0]) & (rt <= q[1]), rt > q[1]]])

stim = np.where(np.random.default_rng(1).random(6000) < 0.5, 1, -1)
v = 0.8 * stim
ch_s, rt_s = ddm(v, lambda t: 1.0, z=0.3)
ch_d, rt_d = ddm(v + 0.4, lambda t: 1.0)
p_start, p_drift = by_tercile(ch_s == 1, rt_s), by_tercile(ch_d == 1, rt_d)
acc_start, acc_drift = (ch_s == stim).mean(), (ch_d == stim).mean()
print("P(right) by tercile, starting-point bias:", np.round(p_start, 3), " accuracy", round(acc_start, 3))
print("P(right) by tercile, drift bias:         ", np.round(p_drift, 3), " accuracy", round(acc_drift, 3))

In [ ]:
import numpy as np
def test_start():
    "with a starting-point bias, the fastest third of responses is far more biased than the slowest"
    assert p_start[0] - p_start[2] > 0.25
def test_drift():
    "with a drift bias, the bias changes less than half as much across terciles"
    assert (p_drift[0] - p_drift[2]) < 0.5 * (p_start[0] - p_start[2])
def test_matched():
    "the two biases give similar overall accuracy"
    assert abs(acc_start - acc_drift) < 0.03

test_start()
test_drift()
test_matched()
print("All checks passed")

### Exercise: Signatures of a collapsing bound

Run the unbiased model with a fixed bound of 1 and with a bound 1.4 exp(-t / 0.8) (floored at 0.05). Compute accuracy by response time tercile and the skewness of each response time distribution.

In [ ]:
def skew(x):
    return ((x - x.mean()) ** 3).mean() / x.std() ** 3

ch_f, rt_f = ...
ch_c, rt_c = ...
acc_fixed, acc_coll = by_tercile(ch_f == stim, rt_f), by_tercile(ch_c == stim, rt_c)
print("accuracy by tercile, fixed bound:     ", np.round(acc_fixed, 3), f" mean RT {rt_f.mean():.2f} s, skew {skew(rt_f):.2f}")
print("accuracy by tercile, collapsing bound:", np.round(acc_coll, 3), f" mean RT {rt_c.mean():.2f} s, skew {skew(rt_c):.2f}")
plt.hist(rt_f, 80, alpha=0.5, density=True, label="fixed"); plt.hist(rt_c, 80, alpha=0.5, density=True, label="collapsing"); plt.legend(); plt.xlabel("decision time (s)"); plt.show()

In [ ]:
import numpy as np
def test_flat():
    "with a fixed bound, accuracy is about the same in fast and slow responses"
    assert abs(acc_fixed[0] - acc_fixed[2]) < 0.04
def test_fall():
    "with a collapsing bound, slow responses are at least 8 points less accurate than fast ones"
    assert acc_coll[0] - acc_coll[2] > 0.08
def test_tail():
    "the collapsing bound cuts the right tail: lower skewness"
    assert skew(rt_c) < 0.6 * skew(rt_f)

test_flat()
test_fall()
test_tail()
print("All checks passed")